In [1]:
# For processing the timeseries
import pandas as pd, os, datetime
import numpy as np

# For plotting
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
import plotly.express as px
pio.renderers.default = 'notebook'

In [2]:
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')

nmap_path = '/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess'
ehf_fpath = '/scratch/ng72/ms5578/time_series'
gen_fpath = '/scratch/ng72/ms5578/time_series/nem_generation'

In [3]:
sdate, edate = '2019-01-06','2019-02-16'

In [4]:
gen_details = pd.read_csv(f"{nmap_path}/gen_details.csv")

In [5]:
hw_tseries = pd.read_csv(f"{ehf_fpath}/gen_hw_status.csv")

In [6]:
def process_group(grp, gen_fpath, hw_tseries, start_date=sdate, end_date=edate):
    """
    Processes a single group of generator data.

    Parameters:
        grp (pd.DataFrame): A single group from gen_details (e.g., from groupby('region')).
        gen_fpath (str): Path to directory containing CSV files named by DUID (e.g., DUID.csv).
        hw_tseries (pd.DataFrame): DataFrame with columns 'time', 'DUID', and data to merge.
        start_date (str): Start date for subsetting the time series.
        end_date (str): End date for subsetting the time series.

    Returns:
        pd.DataFrame: The merged result for the group.
    """
    gen_locs = gen_fpath + '/' + grp['DUID'] + ".csv"
    dfs = [pd.read_csv(fp,dtype='object') for fp in gen_locs if os.path.exists(fp)]
    if not dfs:
        return None

    # This is in case of accidental mid-file headers
    dfs = pd.concat(dfs, ignore_index=True)
    header_row = dfs.columns.tolist()
    dfs = dfs[~dfs.apply(lambda row: list(row) == header_row, axis=1)]

    # This is to correct the column types after removing header rows
    dfs['time'] = pd.to_datetime(dfs['time'])
    dfs['TOTALCLEARED'] = dfs['TOTALCLEARED'].astype(float)
    dfs['TOTALMWh'] = dfs['TOTALMWh'].astype(float)
    dfs['AGCSTATUS'] = pd.to_numeric(dfs['AGCSTATUS'], errors='coerce').fillna(0).astype(int)
    
    dfs['time'] = pd.to_datetime(dfs['time'])
    dfs = dfs.set_index('time').sort_index()
    dfs = dfs.loc[start_date:end_date]

    agg_func = {'TOTALMWh':'sum','TOTALCLEARED':'sum','AGCSTATUS':'min'}
    dfs = dfs.groupby(['DUID', pd.Grouper(freq='1d')]).agg(agg_func)

    hw_tseries['time'] = pd.to_datetime(hw_tseries['time'])
    hw_tseries = hw_tseries.set_index(['time']).sort_index()
    hw_tseries = hw_tseries.loc[sdate:edate]
    hw_tseries = hw_tseries.reset_index().set_index(['DUID','time']).sort_index()

    merged = pd.merge_asof(
        dfs.sort_values(by=['time', 'DUID']),
        hw_tseries.sort_values(by=['time', 'DUID']),
        by='DUID',
        on='time',
        tolerance=pd.Timedelta("12h"),
        direction='nearest'
    )

    return merged

In [7]:
groups = gen_details.groupby('region',
                            as_index = False)

df = groups.get_group('VIC1')

06/01/2019-12/01/2019 BEFORE
13/01/2019-31/01/2019 DURING 
09/02/2019-16/02/2019 AFTER

In [19]:
# This retrieves only the time surrounding the heatwave
df = process_group(df, gen_fpath, hw_tseries, sdate, edate)
df = df.merge(gen_details[['DUID','fuel_source_primary','reg_cap_generation_mw']], left_on='DUID', right_on='DUID', how='left')

In [20]:
df['reg_cap_generation_mw'] = df['reg_cap_generation_mw'].astype('float')
df['cap_norm'] = df['TOTALMWh']/df['reg_cap_generation_mw']

In [21]:
def clean_df(df,t1='00:00',t2='23:59',hw=False,gen_details=gen_details,AGC=False):
    df = df[~((df['cap_norm'] > 2.0) | (df['cap_norm'] <0)) ]
    df = df.drop('reg_cap_generation_mw',axis=1)

    df_sliced = df.loc[(df['time'].dt.month >= 11)| (df['time'].dt.month <= 3)].set_index('time')
    df_sliced = df_sliced.between_time(t1,t2).reset_index()

    if hw == True:
        df_sliced = df_sliced[df_sliced['EHF_flag'] == 1]

    if AGC == True:
        df_sliced = df_sliced[df_sliced['AGCSTATUS'] == 1]
    
    return df_sliced

In [22]:
# df = clean_df(df)

In [23]:
demand = pd.read_csv('/scratch/ng72/ms5578/time_series/state_demand.csv')
demand['time'] = pd.to_datetime(demand['time'])
demand = demand.set_index('time').sort_index()
demand = demand.loc[sdate:edate]
demand = demand[demand["REGIONID"] ==  'VIC1'].drop('REGIONID',axis=1)
demand = demand.resample('d').sum()

In [24]:
gens_in_hw = df.groupby('time').agg({'EHF_flag': 'sum'})

In [28]:
wind = df.copy()[df['fuel_source_primary']=='Wind']
wind['EHF_flag'] = wind['EHF_flag'].astype(str)
# day = wind.set_index('time').loc['2019-01-24':'2019-01-24'].reset_index()
px.line(wind,
          x='time',
          y='TOTALMWh',
          color='DUID')